# 05 - End-to-End Intent Pipeline

This notebook connects the ideas from the earlier notebooks.

Goal: understand how one user message becomes one final intent and one final category.

## The Full Flow

The class deck teaches this pipeline:

```text
raw user message
-> scope/profile resolution
-> context/allowed intent setup
-> dialogue act precheck
-> classification
-> allowed-intent gate
-> post-classification overrides
-> deterministic category derivation
```

Important sentence:

```text
The classifier predicts intent. The backend decides the final safe result.
```

In [3]:
# Step 1: Define the taxonomy.
# These are the intent labels our system understands.

course_info = {
    "course_list",
    "course_details",
    "course_pricing_payment",
    "course_projects",
    "course_schedule_duration",
    "career_certificate_outcomes",
}

fit_eligibility = {
    "prerequisite_fit",
    "beginner_friendliness",
    "background_fit",
    "career_guidance",
}

support_action = {
    "demo_trial_counseling",
    "enrollment_admission",
    "student_support_redirect",
    "instructor_info",
    "recording_access",
    "missed_class_recovery",
}

system_states = {"neutral", "irrelevant", "unknown"} # the upper cases are callled intent . They are created to handle business. but this system_state is created to handle non-business intent. #

all_intents = course_info | fit_eligibility | support_action | system_states

len(all_intents)

19

In [6]:
# Step 2: Resolve the business profile.
# For this project, we only use one profile: EdTech.
# A real system may have many profiles, like ecommerce, banking, healthcare, etc.

edtech_profile = {
    "business_type": "edtech",
    "allowed_intents": all_intents,
}

edtech_profile

{'business_type': 'edtech',
 'allowed_intents': {'background_fit',
  'beginner_friendliness',
  'career_certificate_outcomes',
  'career_guidance',
  'course_details',
  'course_list',
  'course_pricing_payment',
  'course_projects',
  'course_schedule_duration',
  'demo_trial_counseling',
  'enrollment_admission',
  'instructor_info',
  'irrelevant',
  'missed_class_recovery',
  'neutral',
  'prerequisite_fit',
  'recording_access',
  'student_support_redirect',
  'unknown'}}

In [7]:
# Step 3: Detect simple dialogue acts before classification.
# Dialogue act means the message is not always a business question.
# Example: "thanks" is an acknowledgement, not a course inquiry.

def detect_dialogue_act(text):
    text = text.strip().lower()

    if text in {"hi", "hello", "hey", "salam"}:
        return "greeting"

    if text in {"ok", "okay", "thanks", "thank you", "done"}:
        return "acknowledgement"

    return None

detect_dialogue_act("thanks")

'acknowledgement'

In [8]:
# Step 4: Classify with local fallback rules.
# In the production backend, DeepSeek can classify first.
# Here we use local rules so the notebook runs without an API key.

keyword_rules = {
    "course_pricing_payment": ["price", "fee", "fees", "cost", "payment", "koto", "taka"],
    "recording_access": ["record", "recording", "recorded", "lms"],
    "course_schedule_duration": ["schedule", "batch", "duration", "class kobe", "time"],
    "course_list": ["course list", "all course", "courses", "ki ki course", "catalog"],
    "career_guidance": ["confused", "career", "which course", "kon course", "suggest"],
    "demo_trial_counseling": ["human", "counselor", "call me", "phone", "demo", "trial"],
}

def local_classify(text):
    normalized_text = text.lower()

    for intent, keywords in keyword_rules.items():
        if any(keyword in normalized_text for keyword in keywords):
            return {
                "intent": intent,
                "confidence": "high",
                "source": "local",
                "reason": "Matched local keyword rule.",
            }

    return {
        "intent": "unknown",
        "confidence": "low",
        "source": "local",
        "reason": "No local keyword matched.",
    }

local_classify("ML course er price koto?")

{'intent': 'course_pricing_payment',
 'confidence': 'high',
 'source': 'local',
 'reason': 'Matched local keyword rule.'}

In [9]:
# Step 5: Apply the allowed-intent gate.
# If a classifier returns something outside the profile, we do not trust it.

def allowed_intent_gate(classification, profile):
    allowed_intents = profile["allowed_intents"]

    if classification["intent"] not in allowed_intents:
        return {
            "intent": "unknown",
            "confidence": "low",
            "source": classification["source"],
            "reason": f"{classification['intent']} is outside the trusted profile.",
        }

    return classification

fake_wrong_result = {
    "intent": "bank_loan_application",
    "confidence": "high",
    "source": "ai",
    "reason": "Fake example.",
}

allowed_intent_gate(fake_wrong_result, edtech_profile)

{'intent': 'unknown',
 'confidence': 'low',
 'source': 'ai',
 'reason': 'bank_loan_application is outside the trusted profile.'}

In [11]:
# Step 6: Apply post-classification overrides.
# Overrides are manual business rules written by developers/product team.
# They fix cases where the business signal is very obvious.

def apply_overrides(text, classification, dialogue_act=None):
    normalized_text = text.lower()
    intent = classification["intent"]

    if dialogue_act == "acknowledgement":
        classification["intent"] = "neutral"
        classification["confidence"] = "high"
        classification["reason"] = "Acknowledgement override."
        return classification

    if "course list" in normalized_text or "ki ki course" in normalized_text:
        if intent in {"course_list", "career_guidance", "neutral", "unknown"}:
            classification["intent"] = "course_list"
            classification["confidence"] = "high"
            classification["reason"] = "Explicit course list override."
            return classification

    if "confused" in normalized_text or "kon course" in normalized_text:
        if intent in {"neutral", "unknown"}:
            classification["intent"] = "career_guidance"
            classification["confidence"] = "high"
            classification["reason"] = "Decision support override."
            return classification

    return classification

apply_overrides("thanks", local_classify("thanks"), dialogue_act="acknowledgement")

{'intent': 'neutral',
 'confidence': 'high',
 'source': 'local',
 'reason': 'Acknowledgement override.'}

In [12]:
# Step 7: Derive category from final intent.
# Again: category is not predicted by AI.

intent_to_category = {intent: "relevant" for intent in course_info | fit_eligibility | support_action}
intent_to_category.update({
    "neutral": "neutral",
    "irrelevant": "irrelevant",
    "unknown": "unknown",
})

def category_for_intent(intent):
    return intent_to_category.get(intent, "unknown")

category_for_intent("course_pricing_payment")

'relevant'

In [14]:
# Step 8: Run the full pipeline.
# Notice how every stage writes a trace message.
# This makes the system easier to debug and easier to explain in interviews.

def run_pipeline(text, profile=edtech_profile):
    trace = []

    trace.append("1. Scope resolution: resolved EdTech profile.")
    trace.append("2. Context assembly: loaded allowed intents.")

    dialogue_act = detect_dialogue_act(text)
    trace.append(f"3. Dialogue-act precheck: {dialogue_act or 'none'}.")

    classification = local_classify(text)
    trace.append(f"4. Classification: {classification['intent']} from {classification['source']}.")

    classification = allowed_intent_gate(classification, profile)
    trace.append(f"5. Allowed-intent gate: final gate intent is {classification['intent']}.")

    classification = apply_overrides(text, classification, dialogue_act)
    trace.append(f"6. Overrides: intent after overrides is {classification['intent']}.")

    category = category_for_intent(classification["intent"])
    trace.append(f"7. Category derivation: category is {category}.")

    return {
        "text": text,
        "intent": classification["intent"],
        "category": category,
        "confidence": classification["confidence"],
        "source": classification["source"],
        "reason": classification["reason"],
        "trace": trace,
    }

run_pipeline("ML course er price koto?")

{'text': 'ML course er price koto?',
 'intent': 'course_pricing_payment',
 'category': 'relevant',
 'confidence': 'high',
 'source': 'local',
 'reason': 'Matched local keyword rule.',
 'trace': ['1. Scope resolution: resolved EdTech profile.',
  '2. Context assembly: loaded allowed intents.',
  '3. Dialogue-act precheck: none.',
  '4. Classification: course_pricing_payment from local.',
  '5. Allowed-intent gate: final gate intent is course_pricing_payment.',
  '6. Overrides: intent after overrides is course_pricing_payment.',
  '7. Category derivation: category is relevant.']}

In [15]:
# Try several messages and compare the final intent/category.

examples = [
    "ML course er price koto?",
    "recording pabo?",
    "ami confused, kon course nibo?",
    "thanks",
    "ajke weather kemon?",
]

for message in examples:
    result = run_pipeline(message)
    print(message)
    print("  intent:", result["intent"])
    print("  category:", result["category"])
    print("  reason:", result["reason"])
    print()

ML course er price koto?
  intent: course_pricing_payment
  category: relevant
  reason: Matched local keyword rule.

recording pabo?
  intent: recording_access
  category: relevant
  reason: Matched local keyword rule.

ami confused, kon course nibo?
  intent: career_guidance
  category: relevant
  reason: Matched local keyword rule.

thanks
  intent: neutral
  category: neutral
  reason: Acknowledgement override.

ajke weather kemon?
  intent: unknown
  category: unknown
  reason: No local keyword matched.



## What You Should Remember

1. The classifier predicts an intent.
2. The allowed-intent gate checks whether that intent is trusted.
3. Override rules can change the intent when business logic is obvious.
4. Category is derived from the final intent.

Interview answer:

```text
I built the system so the LLM classifies only intent. The backend validates the intent, applies business overrides, and derives category deterministically.
```